# 01 — Preparação e normalização dos dados

Este notebook prepara as transações da cafeteria para os modelos de aprendizado de máquina.

## Conjuntos de dados

- `01_original.csv`: exportação original com **3.636 transações**, registradas entre **01/03/2024 e 23/03/2025**.
- `ref_full_datetime.csv`: cópia de referência da mesma fonte, utilizada para recuperar a hora completa. Na exportação original, o Excel manteve apenas `mm:ss.0` na coluna `datetime`.

As colunas originais são `date`, `datetime`, `cash_type`, `card`, `money` e `coffee_name`. Os 89 valores ausentes em `card` pertencem a pagamentos em dinheiro e não são usados pelos modelos.

## Características produzidas

| Característica | Transformação |
|---|---|
| `money` | normalização min-max para o intervalo `[0, 1]` |
| `weekday` | segunda-feira = 0 até domingo = 6 |
| `is_weekend` | 1 para sábado/domingo; 0 nos demais dias |
| `day_period` | madrugada = 0, manhã = 1, tarde = 2, noite = 3 |
| `hour` | hora inteira entre 0 e 23 |
| `cash_type` | cartão = 0; dinheiro = 1 |
| `coffee_name_*` | codificação one-hot do produto |

`card`, `date` e `datetime` são identificadores ou campos de origem e não entram nos arquivos preparados.

In [1]:
from pathlib import Path

import pandas as pd

SOURCE_PATH = Path("01_original.csv")
REFERENCE_PATH = Path("ref_full_datetime.csv")
OUTPUT_PATH = Path("02_normalized.csv")
EXPECTED_ROWS = 3636
EXPECTED_COLUMNS = ["date", "datetime", "cash_type", "card", "money", "coffee_name"]

original = pd.read_csv(SOURCE_PATH)
reference = pd.read_csv(REFERENCE_PATH)

assert original.columns.tolist() == EXPECTED_COLUMNS
assert reference.columns.tolist() == EXPECTED_COLUMNS
assert len(original) == len(reference) == EXPECTED_ROWS

original_date = pd.to_datetime(original["date"], format="%d/%m/%Y")
reference_date = pd.to_datetime(reference["date"], format="%Y-%m-%d")

# Confirma que os dois arquivos representam as mesmas linhas, na mesma ordem.
assert original_date.equals(reference_date)
for column in ["cash_type", "card", "money", "coffee_name"]:
    pd.testing.assert_series_equal(
        original[column], reference[column], check_names=False, check_dtype=False
    )

print(f"Linhas: {len(original):,}".replace(",", "."))
print(f"Período: {original_date.min():%d/%m/%Y} a {original_date.max():%d/%m/%Y}")
print(f"Cartões ausentes: {original['card'].isna().sum()}")
original.head()

Linhas: 3.636
Período: 01/03/2024 a 23/03/2025
Cartões ausentes: 89


,date,datetime,cash_type,card,money,coffee_name
0,01/03/2024,15:50.5,card,ANON-0000-0000-0001,38.7,Latte
1,01/03/2024,19:22.5,card,ANON-0000-0000-0002,38.7,Hot Chocolate
2,01/03/2024,20:18.1,card,ANON-0000-0000-0002,38.7,Hot Chocolate
3,01/03/2024,46:33.0,card,ANON-0000-0000-0003,28.9,Americano
4,01/03/2024,48:14.6,card,ANON-0000-0000-0004,38.7,Latte


## Transformações

A normalização min-max de `money` preserva a ordem dos valores e os coloca entre 0 e 1. As variáveis categóricas são convertidas para números. A padronização necessária para o K-Means será feita no notebook seguinte, somente após a escolha das características de agrupamento.

In [2]:
df = original.copy()
df["date"] = original_date
df["weekday"] = df["date"].dt.dayofweek
df["is_weekend"] = df["weekday"].isin([5, 6]).astype(int)

full_datetime = pd.to_datetime(reference["datetime"], format="mixed")
assert full_datetime.dt.date.astype(str).equals(reference_date.dt.date.astype(str))
df["hour"] = full_datetime.dt.hour

df["day_period"] = pd.cut(
    df["hour"],
    bins=[-1, 5, 11, 17, 23],
    labels=[0, 1, 2, 3],
).astype(int)

df["cash_type"] = df["cash_type"].map({"card": 0, "cash": 1}).astype(int)

money_min = df["money"].min()
money_max = df["money"].max()
assert money_max > money_min
df["money"] = (df["money"] - money_min) / (money_max - money_min)

coffee_one_hot = pd.get_dummies(df["coffee_name"], prefix="coffee_name", dtype=int)
normalized = pd.concat(
    [
        df[["money", "weekday", "is_weekend", "day_period", "hour", "cash_type"]],
        coffee_one_hot,
    ],
    axis=1,
)

assert normalized.shape == (EXPECTED_ROWS, 14)
assert normalized.isna().sum().sum() == 0
assert normalized["money"].between(0, 1).all()
assert coffee_one_hot.sum(axis=1).eq(1).all()

normalized.to_csv(OUTPUT_PATH, index=False)
print(f"Arquivo salvo: {OUTPUT_PATH} — formato {normalized.shape}")
normalized.head()

Arquivo salvo: 02_normalized.csv — formato (3636, 14)


,money,weekday,is_weekend,day_period,hour,cash_type,coffee_name_Americano,coffee_name_Americano with Milk,coffee_name_Cappuccino,coffee_name_Cocoa,coffee_name_Cortado,coffee_name_Espresso,coffee_name_Hot Chocolate,coffee_name_Latte
0,0.940585,4,0,1,10,0,0,0,0,0,0,0,0,1
1,0.940585,4,0,2,12,0,0,0,0,0,0,0,1,0
2,0.940585,4,0,2,12,0,0,0,0,0,0,0,1,0
3,0.492687,4,0,2,13,0,1,0,0,0,0,0,0,0
4,0.940585,4,0,2,13,0,0,0,0,0,0,0,0,1


## Resultado

O arquivo `02_normalized.csv` contém 14 características numéricas e nenhuma observação ausente. Cada transação possui exatamente uma coluna de café com valor 1.